# Qwen2.5-VL SFT Fine-Tuning (Single T4 / 2xT4 on Kaggle)

This notebook runs Supervised Fine-Tuning (SFT) on Qwen2.5-VL-3B using PRM-verified correct reasoning trajectories.
**Goal**: Enforce structured step-by-step outputs, improve logical reasoning, and reduce visual/arithmetic hallucinations.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')
    print('Successfully loaded HF_TOKEN secrets from Kaggle.')
except Exception as e:
    print('Kaggle secrets not available or skipped.')

if not os.path.exists('prm_project'):
    !git clone https://github.com/yahorlahunovich/prm_project.git
    %cd prm_project
else:
    %cd prm_project
    !git pull

In [ ]:
# Install packages without upgrading pre-installed Kaggle PyTorch
!pip install -q transformers accelerate peft trl datasets qwen-vl-utils

In [ ]:
import json
import os
import torch
from datasets import Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from PIL import Image

# Configuration Flag:
# STRICT_ONLY = True  -> 84 ultra-clean samples (All steps = 1 AND GT match)
# STRICT_ONLY = False -> 300 samples (All GT-matching correct rollouts across all chart types)
STRICT_ONLY = False

# 1. Ensure chart images directory exists & download images if missing
images_dir = 'data/CharXiv/images'
if not os.path.exists(images_dir) or len(os.listdir(images_dir)) == 0:
    print('Images directory empty or missing. Auto-downloading chart images...')
    os.system('python scripts/download_images.py')

# 2. Extract verified correct reasoning trajectories for SFT
evals_path = 'experiments/001_500_reasoning/data/evaluated_rollouts.jsonl'
cleaned_path = 'experiments/001_500_reasoning/data/001_500_reasoning_cleaned.jsonl'

meta = {}
with open(cleaned_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        qid = str(data['question_id'])
        ridx = data['rollout_index']
        gt = str(data['ground_truth']).strip().lower()
        ans = str(data['model_final_answer']).strip().lower()
        is_correct = (gt in ans or ans in gt) and len(ans) > 0
        meta[(qid, ridx)] = {
            'is_correct': is_correct,
            'question': data.get('question', ''),
            'reasoning': data.get('reasoning_steps', '')
        }

sft_conversations = []
with open(evals_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        qid = str(data['question_id'])
        ridx = data['rollout_index']
        evals = data.get('evaluations', [])
        if not evals:
            continue
        all_pass = all(s.get('score') == 1 for s in evals)
        m = meta.get((qid, ridx), {})
        is_correct = m.get('is_correct', False)
        
        # Apply filtering rule based on STRICT_ONLY flag
        keep_sample = (all_pass and is_correct) if STRICT_ONLY else is_correct
        
        if keep_sample and m.get('reasoning'):
            abs_path = os.path.abspath(f'data/CharXiv/images/{qid}.jpg')
            if not os.path.exists(abs_path):
                continue
            
            prompt_text = 'Analyze this chart. Provide step-by-step reasoning and a final answer.\n' + m['question']
            
            messages = [
                {
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'image': abs_path},
                        {'type': 'text', 'text': prompt_text}
                    ]
                },
                {
                    'role': 'assistant',
                    'content': m['reasoning']
                }
            ]
            sft_conversations.append({'messages': messages})

print(f'Extracted {len(sft_conversations)} SFT reasoning trajectories (STRICT_ONLY={STRICT_ONLY}).')
dataset = Dataset.from_list(sft_conversations)
print(dataset)

In [ ]:
# 3. Load Model & Processor in FP16
model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    attn_implementation='sdpa',
    device_map={'': 0}
)

model.enable_input_require_grads()

if hasattr(model, 'visual'):
    model.visual.requires_grad_(False)

processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=512*28*28)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

In [ ]:
# 4. Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM',
)

In [ ]:
# 5. Define SFT Trainer & Train
training_args = SFTConfig(
    output_dir='./sft_qwen_vl',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=5,
    save_steps=25,
    save_total_limit=2,
    gradient_checkpointing=True,
    dataset_num_proc=1,
    remove_unused_columns=False,
    report_to='none',
    dataset_text_field='messages',
    fp16=True,
    bf16=False
)

print('\n=== INITIALIZING SFTTRAINER ===')
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=processor,
    peft_config=peft_config,
)

print('\n=== STARTING SFT TRAINING LOOP ===')
trainer.train()
trainer.save_model('qwen_vl_sft_adapter')
print('\nSFT Training complete! Adapter saved to qwen_vl_sft_adapter.')